<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_32_File_Timestamp_(MACB)_Timestomping_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# Experiment 4: File Timestamp (MACB) Timestomping Detector
# ==========================================================

from datetime import datetime

# Timestamp format
TS_FMT = "%Y-%m-%d %H:%M:%S"


# ----------------------------------------------------------
# Function to detect timestomping
# ----------------------------------------------------------
def detect_timestomping(file_meta, change_gap_minutes=60):
    """
    Detect suspicious timestamp relationships.

    file_meta should contain:
    - born
    - modified
    - accessed
    - changed

    Returns:
        (is_suspicious, reasons)
    """

    modified = datetime.strptime(file_meta["modified"], TS_FMT)
    accessed = datetime.strptime(file_meta["accessed"], TS_FMT)
    changed = datetime.strptime(file_meta["changed"], TS_FMT)
    born = datetime.strptime(file_meta["born"], TS_FMT)

    reasons = []

    # Modified before file creation
    if modified < born:
        reasons.append(
            "Modified time is earlier than Born (creation) time"
        )

    # Accessed before file creation
    if accessed < born:
        reasons.append(
            "Accessed time is earlier than Born (creation) time"
        )

    # Large gap between Changed and Modified timestamps
    gap_minutes = abs((changed - modified).total_seconds()) / 60

    if gap_minutes > change_gap_minutes and changed > modified:
        reasons.append(
            f"MFT Changed time is {gap_minutes:.0f} minutes after Modified time — "
            "metadata may have been altered after the fact"
        )

    return (len(reasons) > 0, reasons)


# ==========================================================
# Test Cases
# ==========================================================

def test_experiment4():

    # Normal file
    normal_file = {
        "born": "2026-01-10 09:00:00",
        "modified": "2026-01-10 09:05:00",
        "accessed": "2026-01-12 14:00:00",
        "changed": "2026-01-10 09:05:00",
    }

    # Tampered file
    tampered_file = {
        "born": "2026-02-01 12:00:00",
        "modified": "2020-01-01 00:00:00",
        "accessed": "2026-02-01 12:00:00",
        "changed": "2026-02-01 12:03:00",
    }

    # Check normal file
    suspicious1, reasons1 = detect_timestomping(normal_file)

    print("Normal File Analysis")
    print("-" * 50)
    print("Suspicious :", suspicious1)
    print("Reasons     :", reasons1)
    print()

    # Check tampered file
    suspicious2, reasons2 = detect_timestomping(tampered_file)

    print("Tampered File Analysis")
    print("-" * 50)
    print("Suspicious :", suspicious2)

    print("Reasons:")
    for reason in reasons2:
        print("-", reason)

    print()

    # Assertions
    assert suspicious1 is False

    assert suspicious2 is True

    assert any(
        "Modified time is earlier" in reason
        for reason in reasons2
    )

    print("All test cases passed.")


# ----------------------------------------------------------
# Run Test
# ----------------------------------------------------------

test_experiment4()

Normal File Analysis
--------------------------------------------------
Suspicious : False
Reasons     : []

Tampered File Analysis
--------------------------------------------------
Suspicious : True
Reasons:
- Modified time is earlier than Born (creation) time
- MFT Changed time is 3201843 minutes after Modified time — metadata may have been altered after the fact

All test cases passed.
